# M3: zero-shot Whisper runs on the 1,000-clip test set (Kaggle)

Each entry in `RUNS` predicts with `models.whisper_full.zero_shot`, then scores with `common.score`.
Each run's folder (`m3-<model>-zeroshot-...`) is copied to `/kaggle/working/` as soon as it finishes. Its proof is:
- `predictions.csv`, `predict.log`, `scores.json` and the environment file;
- the raw output in `stdout-*.log`.

Download the folders into `eval/results/` and commit them. Steps: `knowledge/how-to-train-on-kaggle.md`.

In [ ]:
# 0. Settings: the only cell to change between runs
RUNS = [  # model, extra flags
    ("openai/whisper-tiny", "--fallback"),
    ("openai/whisper-base", "--fallback"),
    ("openai/whisper-small", "--fallback --detect"),
]
BRANCH = "6-whisper"
W = "/kaggle/working"


In [ ]:
# 1. Check the machine: GPU, disk. Saved as proof.
!(date -u; nvidia-smi; df -h /kaggle/working /tmp) | tee {W}/machine-$(date -u +%Y%m%dT%H%M%S).log

In [ ]:
# 2. Get the code at the branch, install the locked dependencies
!rm -rf /tmp/repo && git clone --quiet --branch {BRANCH} https://github.com/ASU-MICS-2028/nlp-group-1-prosit-3.git /tmp/repo
!pip install --quiet uv
!cd /tmp/repo && git log -1 --format='commit %H %s' && uv sync --frozen --quiet && uv run python -c "import torch; print('cuda', torch.cuda.is_available(), torch.__version__)"

In [ ]:
# 3. Predict and score each run; copy its folder to /kaggle/working as soon as it is done
import subprocess, shutil, glob
for model, flags in RUNS:
    tag = model.split("/")[-1] + "-zeroshot" + ("-fallback" if "--fallback" in flags else "") + ("-detect" if "--detect" in flags else "")
    out = f"/tmp/repo/eval/results/m3-{tag}"
    print("=====", model, flags, "->", out, flush=True)
    log = f"{W}/stdout-{tag}.log"
    cmd = (f"cd /tmp/repo && uv run python -m models.whisper_full.zero_shot {model} {flags} 2>&1 | tee -a {log} && "
           f"uv run python -m common.score {out}/predictions.csv 2>&1 | tee -a {log}")
    subprocess.run(cmd, shell=True)
    shutil.copy(log, out + "/stdout-predict-and-score.log") if glob.glob(out) else None
    shutil.copytree(out, f"{W}/m3-{tag}", dirs_exist_ok=True) if glob.glob(out) else print("NO OUTPUT for", tag)

In [ ]:
# 4. What was saved, and the overall score of each run
!ls -la {W}; grep -h "overall" {W}/stdout-*.log